# Deep Visual Style Recognition Across Domains

### A deep learning study of cross-domain art-versus-photo classification

## Abstract

This project extends a previous classical machine learning study on visual style classification by moving from handcrafted visual descriptors to deep learned representations. The earlier Visual Style Classification with Handcrafted Features project showed that handcrafted features such as HOG, LBP, and Gabor filters can achieve strong performance in a constructed WikiArt-versus-Unsplash classification setting. However, the analysis also revealed an important limitation: source-domain bias. A metadata-only diagnostic showed that technical image properties can strongly separate the datasets, meaning that high classification performance may partly reflect dataset-specific artefacts rather than generalizable visual style patterns.

This project addresses that limitation by using a multi-source dataset design and deep learning models to evaluate whether learned visual representations generalize better across domains. The project compares a convolutional neural network trained from scratch with transfer learning based on pretrained CNN architectures, with a primary focus on ResNet18. The task remains a binary classification problem between artistic images and real-world photographic images, but the data is expanded to include multiple independent sources for each class.

Model performance is evaluated under both random stratified splits and stricter cross-dataset splits, where models are trained on one pair of art/photo sources and tested on unseen sources. This makes it possible to distinguish between closed-domain performance and cross-domain generalization. In addition, metadata-only diagnostics, learning curves, confusion matrices, embedding visualizations, and Grad-CAM analysis are used to interpret model behaviour and inspect whether the models rely on meaningful visual structure or dataset-specific artefacts.

The central goal is not only to improve classification performance, but to investigate whether deep learned visual representations can generalize beyond the original source-biased experimental setup and provide a stronger basis for art-versus-photo classification across domains.

## Introduction

Deep learning has become the dominant approach in modern computer vision because convolutional neural networks can learn visual representations directly from image data. Unlike classical machine learning pipelines, which depend on manually designed features such as entropy, edge density, HOG, LBP, or Gabor filters, deep learning models can learn hierarchical representations of visual structure from pixels.

This project builds directly on the findings of the previous Visual Style Classification with Handcrafted Features project, a classical machine learning study focused on distinguishing artistic images from natural photographs using handcrafted visual descriptors. That project demonstrated that compact texture- and frequency-based descriptors can perform surprisingly well. In particular, LBP and Gabor feature fusion achieved strong classification performance while using far fewer dimensions than HOG. However, the project also identified a major limitation: the WikiArt-versus-Unsplash dataset construction introduced source-domain bias. The classifier may have learned technical differences between datasets rather than purely general visual differences between artwork and photography.

The present project therefore asks a new question: can deep learned representations improve cross-domain generalization in visual style classification?

To answer this question, the project uses a multi-source dataset design. The artistic class is represented by images from WikiArt and ArtBench, while the photographic class is represented by images from Places365 and a COCO subset. This design reduces dependence on a single source per class and enables cross-dataset evaluation. Models are first evaluated on a standard random stratified split and then on stricter cross-dataset splits, where training and testing sources are intentionally separated.

The project compares three main modelling approaches: a metadata-only baseline for bias diagnostics, a CNN trained from scratch, and transfer learning using a pretrained ResNet18 model. The final analysis focuses not only on accuracy and ROC-AUC, but also on generalization gaps, error patterns, feature embeddings, and qualitative Grad-CAM explanations.

This project is therefore not simply a deep learning replacement for the previous ML pipeline. It is a continuation of the same research question under a stronger experimental design: from handcrafted representations in a closed domain to learned representations evaluated across domains.

## Research Question and Hypotheses

The previous handcrafted-feature project showed that strong classification performance can be achieved in a closed WikiArt-versus-photo setup. However, it also revealed that dataset-specific technical artefacts may strongly influence the task. Therefore, the central question of this project is not only whether a deep learning model can classify artistic and photographic images, but whether it can generalize across unseen visual data sources.

The main research question is:

**Can deep learned visual representations improve cross-dataset generalization in art-versus-photo classification compared to a source-biased handcrafted-feature setup?**

This question is evaluated through the following hypotheses:

**H1 — Metadata bias hypothesis:**  
Technical image metadata such as resolution, aspect ratio, file size, and file format may contain strong source-domain signal. If metadata-only models achieve high performance, random split results must be interpreted carefully.

**H2 — Transfer learning hypothesis:**  
A pretrained CNN, especially ResNet18, is expected to outperform a CNN trained from scratch because it starts from general visual representations learned from large-scale image data.

**H3 — Cross-dataset generalization hypothesis:**  
Random stratified split performance is expected to be higher than cross-dataset performance. Cross-dataset evaluation is therefore used as a stricter test of whether the model learns transferable visual structure rather than source-specific artefacts.

**H4 — Interpretability hypothesis:**  
Grad-CAM and error analysis can provide qualitative evidence about whether model decisions are influenced by meaningful visual regions or by dataset-specific visual artefacts.

## Dataset Design

The project uses a balanced multi-source dataset design. Instead of assigning each class to a single dataset source, each class is represented by two independent image sources. This is important because if one class comes from only one dataset and the other class comes from only one different dataset, the classification task may become a dataset-identification task rather than a visual-style classification task.

The artistic class consists of:

- **WikiArt** — artistic images from multiple artists, periods, and styles;
- **ArtBench** — a structured artwork benchmark dataset with multiple artistic styles.

The photographic class consists of:

- **Places365** — real-world scene photographs;
- **COCO** — real-world photographic images from everyday scenes.

For the initial full experiment, the project samples 10,000 images from each source dataset, resulting in 40,000 images in total:

| Source dataset | Class | Number of images |
|---|---:|---:|
| WikiArt | art | 10,000 |
| ArtBench | art | 10,000 |
| Places365 | photo | 10,000 |
| COCO | photo | 10,000 |

This produces a balanced binary classification dataset with 20,000 artistic images and 20,000 photographic images.

The goal of this design is not to remove all possible bias, which is unrealistic for real-world image datasets, but to reduce dependence on a single source per class and make cross-dataset evaluation possible.

## Dataset Manifest and Split Strategy

A central dataset manifest is created to make the data selection and split assignment reproducible. The manifest stores the image path, binary label, class name, source dataset, original metadata where available, and the split assignment for each experiment.

The manifest contains the following key columns:

| Column | Meaning |
|---|---|
| `image_path` | Relative path to the image file |
| `label` | Binary target: 1 for art, 0 for photo |
| `class_name` | Human-readable class name: art or photo |
| `source_dataset` | Original dataset source: WikiArt, ArtBench, Places365, or COCO |
| `original_split` | Original train/test split when available, mainly for ArtBench |
| `original_style` | Original ArtBench style category when available |
| `split_random` | Standard random train/validation/test split |
| `split_cross_a` | Cross-dataset split A |
| `split_cross_b` | Cross-dataset split B |

Three split strategies are used.

### Random stratified split

The random split mixes all four source datasets while preserving class and source balance. It provides a standard closed-domain benchmark.

### Cross-dataset split A

In this setting, the model is trained on WikiArt and Places365 and tested on ArtBench and COCO.

| Role | Sources |
|---|---|
| Train/validation | WikiArt + Places365 |
| Test | ArtBench + COCO |

### Cross-dataset split B

In this setting, the model is trained on ArtBench and COCO and tested on WikiArt and Places365.

| Role | Sources |
|---|---|
| Train/validation | ArtBench + COCO |
| Test | WikiArt + Places365 |

The cross-dataset splits are the central evaluation setup of this project. They test whether models can generalize to unseen sources instead of only performing well when the same dataset sources appear in both training and test data.

## Metadata Bias Diagnostic

Before training deep learning models, the project performs a metadata-only diagnostic. The purpose of this step is to test whether technical image properties alone can predict the class label or the source dataset.

The metadata features include:

- image width;
- image height;
- aspect ratio;
- file size;
- log-transformed file size;
- megapixels;
- number of channels;
- landscape/portrait/square indicators;
- file extension.

These features do not describe semantic image content. They only describe technical properties of the image files. Therefore, strong metadata-only performance indicates possible source-domain bias.

The metadata-only results show that the random split contains a very strong technical signal. A Random Forest classifier trained only on metadata reaches very high performance for art-versus-photo classification under the random split. The same metadata features also predict the source dataset with very high accuracy.

However, the cross-dataset metadata experiments behave very differently. When the metadata model is trained on one pair of sources and tested on another pair, performance collapses. This suggests that the metadata model is not learning a stable art-versus-photo distinction, but source-specific technical patterns.

This finding is important for the rest of the project. It shows that random split performance alone is not sufficient. Deep learning models must also be evaluated under cross-dataset splits to test whether they learn transferable visual representations rather than source-specific artefacts.